# Upper Confidence Bound BO via a Remote HTTPGenerator

This example shows how to run a generator on a remote server through `HTTPGenerator`, instead of running it in-process. `HTTPGenerator` is a drop-in `Generator` that forwards `suggest`/`ingest` calls over HTTP to a stateful session on a `xopt.generators.remote.server` instance, which reconstructs the wrapped generator (here, Upper Confidence Bound) from its registered name and config.

For this self-contained example we start the reference server in a background thread, but in practice it would run as its own process/service.

In [ ]:
import os
import socket
import threading
import time

import requests
import uvicorn

from xopt import Xopt, VOCS, Evaluator
from xopt.generators.remote.http import HTTPGenerator
from xopt.generators.remote.server import app
from xopt.resources.test_functions.sinusoid_1d import evaluate_sinusoid

# Ignore all warnings
import warnings

warnings.filterwarnings("ignore")

SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in ["1", "true", "yes"]
N_STEPS = 3 if SMOKE_TEST else 8

## Start the reference HTTP server

Normally this process runs separately (e.g. `python -m xopt.generators.remote.server`). Here we launch it in a background thread on a free local port so the notebook is self-contained.

In [ ]:
def _free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


port = _free_port()
config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

base_url = f"http://127.0.0.1:{port}"
for _ in range(50):
    try:
        requests.get(base_url + "/openapi.json", timeout=0.2)
        break
    except requests.ConnectionError:
        time.sleep(0.1)
else:
    raise RuntimeError("reference server did not start in time")

print(f"Reference server running at {base_url}")

## Build the remote generator and Xopt object

`HTTPGenerator` wraps an already-instantiated generator (here, `UpperConfidenceBoundGenerator`) plus the location of the server (`base_url`). Its full state (including `vocs`) is sent to the server when the session is first created, which reconstructs an instance of the same class there - never by sending a pickled object over the wire.

In [ ]:
from xopt.generators.bayesian.upper_confidence_bound import (
    UpperConfidenceBoundGenerator,
)

vocs = VOCS(
    variables={"x1": [0, 6.28]},
    objectives={"y1": "MINIMIZE"},
)

generator = HTTPGenerator(
    base_url=base_url,
    generator=UpperConfidenceBoundGenerator(vocs=vocs, beta=0.1),
)

evaluator = Evaluator(function=evaluate_sinusoid)
X = Xopt(generator=generator, evaluator=evaluator)
X

## Run optimization

Each `X.step()` call transparently sends the current data to the remote session and asks it to suggest the next point — the optimization loop is identical to running the generator in-process.

In [ ]:
X.random_evaluate(3)
for i in range(N_STEPS):
    print(i)
    X.step()

## View output data

In [ ]:
X.data

## Visualize the remote model

`visualize_model` isn't part of the remote proxy's interface (only `suggest`/`ingest` are). Accessing it triggers `HTTPGenerator` to pull a full, fitted snapshot of the remote generator (including the GP model) and reconstruct a real local copy, then delegate the call to it. This mirror is a read-only, point-in-time snapshot — call `X.generator.pull()` again after more iterations to refresh it.

In [ ]:
fig, ax = X.generator.visualize_model(n_grid=100)

In [ ]:
X.generator.data

## Clean up

Close the remote session and stop the local reference server.

In [ ]:
X.generator.finalize()
server.should_exit = True
server_thread.join(timeout=5)